# TMJ Position Classifier — Training (v3, balanced + augmented)

3D CNN для классификации положения головок ВНЧС по данным КЛКТ.

**Задача:** для каждого сустава (отдельный 3D-кроп) предсказать 2 метки — сагиттальное и фронтальное положение, каждая — один из 3 классов.

**Пайплайн:**
1. **Preprocessing (один раз):** DICOM → детектор находит центры ВНЧС → вырезаем 128³ кропы → сохраняем `.npy` на Drive
2. **Training:** читаем готовые `.npy` кропы (~1 МБ каждый) → 3D CNN с 2 головами

**Изменения v3 vs v2:**
- Weighted CrossEntropyLoss (веса обратно пропорциональны частоте класса)
- Label smoothing 0.1
- Расширенные аугментации: 3D-flips, 90°-повороты, гауссов шум, яркость
- Лёгкая модель: backbone [8, 16, 32, 64] вместо [16, 32, 64, 128]

**Требования:**
- GPU runtime (T4 или лучше): Runtime → Change runtime type → T4 GPU
- Данные на Google Drive в папке `tmj_data/`
- Веса детектора `best_detector.pth` в `tmj_data/models/` (для предобработки)

## 1. Setup

In [1]:
!pip install -q --upgrade scipy tqdm "pydicom>=3.0" "pylibjpeg[libjpeg]>=2.0"


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
import os
import gc
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/tmj_data")

DATASET_ROOT = str(DRIVE_ROOT / "dataset_public")
LABELS_PATH = str(DRIVE_ROOT / "tmj_position_labels.json")
MANIFEST_PATH = str(DRIVE_ROOT / "dataset_public" / "manifest_private.json")

# Веса детектора — нужны для однократной предобработки (секция 3).
DETECTOR_PATH = str(DRIVE_ROOT / "models" / "best_detector.pth")

# Кропы суставов сохраняем на Drive, чтобы не пересобирать при перезапуске сессии.
CROPS_DIR = DRIVE_ROOT / "tmj_crops"
CROPS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = "/content/experiments"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DETECTOR_DS_FACTOR = 6   # downsample для детектора (как при его обучении)
CROP_SIZE = 128           # размер кропа вокруг каждого сустава
CROP_DS_FACTOR = 2        # downsample кропа для обучения: 128 → 64

assert Path(DATASET_ROOT).exists(), f"Dataset not found: {DATASET_ROOT}"
assert Path(LABELS_PATH).exists(), f"Labels not found: {LABELS_PATH}"
assert Path(MANIFEST_PATH).exists(), f"Manifest not found: {MANIFEST_PATH}"

studies = list(Path(DATASET_ROOT).glob("study_*"))
print(f"Found {len(studies)} study folders")
print(f"Detector weights: {Path(DETECTOR_PATH).exists()}")
print(f"Crops dir: {CROPS_DIR}")

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU detected. Training will be very slow.")

print(f"Device: {device}")

## 2. Label Table

Джойн `manifest_private.json` (study_id ↔ patient_name) и `tmj_position_labels.json` (patient → метки).

Сплит строго по пациентам, чтобы исключить утечку данных.

In [ ]:
import json
import random
import logging
from typing import Dict, List, Optional, Tuple

logger = logging.getLogger("tmj_training")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


def map_sagittal(code: int) -> int:
    """Sagittal code 1-3 -> class index 0-2."""
    if code not in (1, 2, 3):
        raise ValueError(f"Invalid sagittal code: {code!r} (expected 1-3)")
    return code - 1


def map_frontal(code: int) -> int:
    """Frontal code 4-6 -> class index 0-2."""
    if code not in (4, 5, 6):
        raise ValueError(f"Invalid frontal code: {code!r} (expected 4-6)")
    return code - 4


def build_index(
    manifest_path: str,
    labels_path: str,
    dataset_root: str,
) -> List[Dict]:
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    with open(labels_path, "r", encoding="utf-8") as f:
        labels_data = json.load(f)

    label_by_name = {}
    for patient in labels_data["patients"]:
        name = patient["name_raw"].strip()
        label_by_name[name] = patient["labels"]

    records = []
    skipped = 0

    for study in manifest["studies"]:
        patient_name = study["patient_name"].strip()
        if patient_name not in label_by_name:
            skipped += 1
            continue

        lbl = label_by_name[patient_name]
        dicom_dir = os.path.join(dataset_root, study["study_id"])

        records.append({
            "study_id": study["study_id"],
            "dicom_dir": dicom_dir,
            "patient_name": patient_name,
            "sag_right": map_sagittal(lbl["sagittal"]["right"]),
            "sag_left":  map_sagittal(lbl["sagittal"]["left"]),
            "fr_right":  map_frontal(lbl["frontal"]["right"]),
            "fr_left":   map_frontal(lbl["frontal"]["left"]),
        })

    logger.info(f"build_index: {len(records)} records matched, {skipped} skipped")
    return records


def split_by_patient(
    records: List[Dict],
    split_ratio: float = 0.8,
    seed: int = 42,
) -> Tuple[List[Dict], List[Dict]]:
    patients = sorted(set(r["patient_name"] for r in records))
    n = len(patients)
    if n == 0:
        return [], []
    if n == 1:
        logger.warning("Only one patient — all records go to train")
        return records, []

    rng = random.Random(seed)
    rng.shuffle(patients)

    split_idx = int(n * split_ratio)
    split_idx = min(max(1, split_idx), n - 1)
    train_patients = set(patients[:split_idx])

    train_records = [r for r in records if r["patient_name"] in train_patients]
    val_records = [r for r in records if r["patient_name"] not in train_patients]

    logger.info(
        f"Split: train={len(train_records)} ({len(train_patients)} patients) / "
        f"val={len(val_records)} ({n - len(train_patients)} patients)"
    )
    return train_records, val_records

In [ ]:
all_records = build_index(MANIFEST_PATH, LABELS_PATH, DATASET_ROOT)
train_records, val_records = split_by_patient(all_records, split_ratio=0.8)

print(f"\nTotal records:  {len(all_records)}")
print(f"Train records:  {len(train_records)}")
print(f"Val records:    {len(val_records)}")

if all_records:
    print(f"\nSample record: {all_records[0]}")

## 3. Preprocessing: Detect → Crop → Save (один раз)

Детектор находит центры левого и правого ВНЧС, вырезаем 128³ куб вокруг каждого, нормализуем и сохраняем `.npy` на Drive.

**Запускать только один раз** — при повторных сессиях кропы уже на Drive.

In [ ]:
import numpy as np
import pydicom
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


# --------------- TMJDetectorLarge (копия из MLService/models/tmj_detector.py) ---------------

class TMJDetectorLarge(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.conv1 = self._cb(in_channels, 32);  self.pool1 = nn.MaxPool3d(2)
        self.conv2 = self._cb(32, 64);            self.pool2 = nn.MaxPool3d(2)
        self.conv3 = self._cb(64, 128);           self.pool3 = nn.MaxPool3d(2)
        self.conv4 = self._cb(128, 256);          self.pool4 = nn.MaxPool3d(2)
        self.conv5 = self._cb(256, 512)
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.fc_left  = nn.Sequential(nn.Linear(512,256), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(256,3), nn.Sigmoid())
        self.fc_right = nn.Sequential(nn.Linear(512,256), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(256,3), nn.Sigmoid())

    @staticmethod
    def _cb(ic, oc):
        return nn.Sequential(
            nn.Conv3d(ic,oc,3,padding=1), nn.BatchNorm3d(oc), nn.ReLU(True),
            nn.Conv3d(oc,oc,3,padding=1), nn.BatchNorm3d(oc), nn.ReLU(True),
        )

    def forward(self, x):
        x = self.pool1(self.conv1(x)); x = self.pool2(self.conv2(x))
        x = self.pool3(self.conv3(x)); x = self.pool4(self.conv4(x))
        x = self.conv5(x)
        x = self.global_pool(x).view(x.size(0), -1)
        return torch.cat([self.fc_left(x), self.fc_right(x)], dim=1)


# --------------- Streaming preprocessing (low-memory) ---------------

def _sort_dicom_files(dicom_dir: str) -> List[Path]:
    """Read only DICOM headers (no pixels) to sort files by position."""
    dcm_files = sorted(Path(dicom_dir).glob("*.dcm"))
    if not dcm_files:
        raise FileNotFoundError(f"No .dcm files in {dicom_dir}")

    entries = []
    for f in dcm_files:
        hdr = pydicom.dcmread(str(f), stop_before_pixels=True)
        try:
            key = float(hdr.ImagePositionPatient[2])
        except (AttributeError, IndexError, TypeError):
            key = float(hdr.InstanceNumber)
        entries.append((key, f))
    entries.sort(key=lambda e: e[0])
    return [e[1] for e in entries]


def _read_one_slice_hu(path: Path) -> np.ndarray:
    """Read a single DICOM file → HU-corrected float32 2D array."""
    import pylibjpeg  # noqa: F401
    ds = pydicom.dcmread(str(path))
    arr = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    return arr * slope + intercept


def _normalize_inplace(vol: np.ndarray) -> np.ndarray:
    p2, p98 = np.percentile(vol, [2, 98])
    np.clip(vol, p2, p98, out=vol)
    d = p98 - p2
    if d > 0:
        vol -= p2
        vol /= d
    else:
        vol[:] = 0.0
    return vol


def detect_centers_streaming(
    sorted_files: List[Path],
    detector: nn.Module,
    ds_factor: int,
    device: str,
) -> Dict[str, np.ndarray]:
    """
    Pass 1: read slices one-by-one, downsample (H,W) immediately,
    build a small volume for the detector.  Peak RAM ~15 MB.
    """
    small_slices = []
    orig_H = orig_W = None

    for fpath in sorted_files:
        arr = _read_one_slice_hu(fpath)  # (H, W) full-res
        if orig_H is None:
            orig_H, orig_W = arr.shape
        tgt_h, tgt_w = orig_H // ds_factor, orig_W // ds_factor
        small = ndimage.zoom(arr, (tgt_h / arr.shape[0], tgt_w / arr.shape[1]), order=1)
        small_slices.append(small.astype(np.float32))
        del arr

    vol_hw = np.stack(small_slices, axis=0)  # (D, H/6, W/6) ~14 MB
    del small_slices

    D_orig = vol_hw.shape[0]
    D_ds = D_orig // ds_factor
    vol_ds = ndimage.zoom(vol_hw, (D_ds / D_orig, 1.0, 1.0), order=1).astype(np.float32)
    del vol_hw

    _normalize_inplace(vol_ds)

    t = torch.from_numpy(vol_ds[None, None]).float().to(device)
    with torch.no_grad():
        pred = detector(t).cpu().numpy()[0]
    del t, vol_ds
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    nD, nH, nW = D_ds, orig_H // ds_factor, orig_W // ds_factor
    left  = (np.array([pred[0]*nD, pred[1]*nH, pred[2]*nW]) * ds_factor).astype(int)
    right = (np.array([pred[3]*nD, pred[4]*nH, pred[5]*nW]) * ds_factor).astype(int)
    return {"left": left, "right": right}


def extract_crop_streaming(
    sorted_files: List[Path],
    center: np.ndarray,
    crop_size: int,
) -> np.ndarray:
    """
    Pass 2: read ONLY the slices needed for the crop, extract (y,x) region
    from each one.  Peak RAM ~8 MB for 128^3.
    """
    total_z = len(sorted_files)
    half = crop_size // 2
    cz, cy, cx = int(center[0]), int(center[1]), int(center[2])

    z_start = max(0, cz - half)
    z_end   = min(total_z, cz + half)

    planes = []
    for idx in range(z_start, z_end):
        arr = _read_one_slice_hu(sorted_files[idx])
        H, W = arr.shape
        y_s, y_e = max(0, cy - half), min(H, cy + half)
        x_s, x_e = max(0, cx - half), min(W, cx + half)
        planes.append(arr[y_s:y_e, x_s:x_e])
        del arr

    crop = np.stack(planes, axis=0).astype(np.float32)
    del planes

    if crop.shape != (crop_size, crop_size, crop_size):
        padded = np.zeros((crop_size, crop_size, crop_size), dtype=np.float32)
        pz = (crop_size - crop.shape[0]) // 2
        py = (crop_size - crop.shape[1]) // 2
        px = (crop_size - crop.shape[2]) // 2
        padded[pz:pz+crop.shape[0], py:py+crop.shape[1], px:px+crop.shape[2]] = crop
        crop = padded

    return _normalize_inplace(crop)


def preprocess_all_studies(
    records: List[Dict],
    detector_path: str,
    crops_dir: Path,
    crop_size: int = 128,
    ds_factor: int = 6,
) -> None:
    """
    Streaming preprocessing: two lightweight passes per study.
    Pass 1 — detect TMJ centers (~15 MB peak).
    Pass 2 — extract & save crops (~8 MB peak).
    """
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    ckpt = torch.load(detector_path, map_location="cpu", weights_only=False)
    detector = TMJDetectorLarge()
    detector.load_state_dict(ckpt["model_state_dict"])
    detector.eval().to(dev)
    del ckpt
    gc.collect()
    logger.info(f"Detector loaded on {dev}")

    crops_dir = Path(crops_dir)
    crops_dir.mkdir(parents=True, exist_ok=True)
    skipped, done = 0, 0

    for rec in tqdm(records, desc="Detect & crop"):
        left_path  = crops_dir / f"{rec['study_id']}_left.npy"
        right_path = crops_dir / f"{rec['study_id']}_right.npy"
        if left_path.exists() and right_path.exists():
            skipped += 1
            continue

        sorted_files = _sort_dicom_files(rec["dicom_dir"])

        centers = detect_centers_streaming(sorted_files, detector, ds_factor, dev)
        gc.collect()

        for side in ("left", "right"):
            out_path = crops_dir / f"{rec['study_id']}_{side}.npy"
            crop = extract_crop_streaming(sorted_files, centers[side], crop_size)
            np.save(out_path, crop)
            del crop

        done += 1
        gc.collect()
        logger.info(f"  {rec['study_id']}: L={centers['left']} R={centers['right']}")

    logger.info(f"Preprocessing done: {done} new, {skipped} cached, total {len(records)}")

Выполните ячейку ниже **один раз**. При повторных запусках она пропускает уже нарезанные кропы.

In [ ]:
preprocess_all_studies(
    all_records,
    detector_path=DETECTOR_PATH,
    crops_dir=CROPS_DIR,
    crop_size=CROP_SIZE,
    ds_factor=DETECTOR_DS_FACTOR,
)

In [ ]:
## 4. Dataset & DataLoader

HEAD_NAMES = ["sagittal", "frontal"]


def expand_to_crop_records(
    study_records: List[Dict], crops_dir: Path,
) -> List[Dict]:
    """study-level records → per-condyle records (2× больше сэмплов)."""
    out = []
    for r in study_records:
        for side in ("left", "right"):
            crop_path = crops_dir / f"{r['study_id']}_{side}.npy"
            if not crop_path.exists():
                continue
            out.append({
                "study_id": r["study_id"],
                "patient_name": r["patient_name"],
                "side": side,
                "crop_path": str(crop_path),
                "sag": r[f"sag_{side}"],
                "fr":  r[f"fr_{side}"],
            })
    return out


class TMJCropDataset(Dataset):
    """Читает готовые .npy кропы (128³), опционально downsample до target_size."""

    def __init__(
        self, records: List[Dict], target_size: int = 64,
        is_train: bool = True, shift_voxels: int = 3,
    ):
        self.records = records
        self.target_size = target_size
        self.is_train = is_train
        self.shift_voxels = shift_voxels
        logger.info(
            f"CropDataset: {len(records)} samples ({'train' if is_train else 'val'}) "
            f"target_size={target_size}"
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        vol = np.load(rec["crop_path"]).astype(np.float32)

        if self.target_size and self.target_size != vol.shape[0]:
            f = self.target_size / vol.shape[0]
            vol = ndimage.zoom(vol, [f, f, f], order=1)

        if self.is_train:
            if self.shift_voxels > 0:
                sv = self.shift_voxels
                for ax in range(3):
                    vol = np.roll(vol, random.randint(-sv, sv), axis=ax)

            for ax in range(3):
                if random.random() < 0.5:
                    vol = np.flip(vol, axis=ax).copy()

            k = random.randint(0, 3)
            if k > 0:
                axes_pair = random.choice([(0, 1), (0, 2), (1, 2)])
                vol = np.rot90(vol, k=k, axes=axes_pair).copy()

            vol = vol + np.random.normal(0, 0.02, vol.shape).astype(np.float32)
            vol *= random.uniform(0.9, 1.1)
            np.clip(vol, 0, 1, out=vol)

        vol_t = torch.from_numpy(np.ascontiguousarray(vol)).float().unsqueeze(0)
        lbl_t = torch.tensor([rec["sag"], rec["fr"]], dtype=torch.long)
        return vol_t, lbl_t


# ---- Expand records & build loaders ----
BATCH_SIZE = 4
TARGET_SIZE = CROP_SIZE // CROP_DS_FACTOR   # 128 // 2 = 64

train_crop_records = expand_to_crop_records(train_records, CROPS_DIR)
val_crop_records   = expand_to_crop_records(val_records,   CROPS_DIR)

train_ds = TMJCropDataset(train_crop_records, target_size=TARGET_SIZE, is_train=True)
val_ds   = TMJCropDataset(val_crop_records,   target_size=TARGET_SIZE, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds)} crops ({len(train_records)} studies × 2)")
print(f"Val:   {len(val_ds)} crops ({len(val_records)} studies × 2)")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Input volume:  (1, {TARGET_SIZE}, {TARGET_SIZE}, {TARGET_SIZE})")

In [ ]:
import matplotlib.pyplot as plt

sample_vol, sample_lbl = train_ds[0]
rec0 = train_crop_records[0]
print(f"Volume shape: {sample_vol.shape}")
print(f"Labels: sagittal={sample_lbl[0].item()}, frontal={sample_lbl[1].item()}")
print(f"Side: {rec0['side']}, study: {rec0['study_id']}")
print(f"Value range: [{sample_vol.min():.3f}, {sample_vol.max():.3f}]")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
mid = sample_vol.shape[1] // 2
axes[0].imshow(sample_vol[0, mid], cmap="gray"); axes[0].set_title(f"Axial ({mid})")
axes[1].imshow(sample_vol[0, :, mid], cmap="gray"); axes[1].set_title(f"Coronal ({mid})")
axes[2].imshow(sample_vol[0, :, :, mid], cmap="gray"); axes[2].set_title(f"Sagittal ({mid})")
for ax in axes:
    ax.axis("off")
fig.suptitle(f"{rec0['side']} TMJ — sag={sample_lbl[0].item()}, fr={sample_lbl[1].item()}")
plt.tight_layout()
plt.show()

## 5. Model

**TMJCondyleClassifier** — лёгкая 3D CNN с **2 головами** (sagittal + frontal).

Каждый сэмпл — один кроп вокруг одного сустава, модели не нужно различать лево/право.

v3: уменьшена ёмкость backbone [8→16→32→64] и повышен dropout (0.5) для борьбы с переобучением на малом датасете.

```
backbone (4 × ConvBlock3d → MaxPool3d): [8, 16, 32, 64]
  → AdaptiveAvgPool3d(1) → (B, 64)
  → head_sag  Linear(64→64) → ReLU → Dropout(0.5) → Linear(64→3)
  → head_fr   Linear(64→64) → ReLU → Dropout(0.5) → Linear(64→3)
```

In [ ]:
def _conv_block(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm3d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm3d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool3d(kernel_size=2, stride=2),
    )


class TMJCondyleClassifier(nn.Module):
    """Классификатор положения одного сустава: 2 головы × 3 класса."""
    NUM_CLASSES = 3

    def __init__(
        self,
        in_channels: int = 1,
        features: Optional[List[int]] = None,
        fc_hidden: int = 64,
        dropout: float = 0.5,
    ):
        super().__init__()
        if features is None:
            features = [8, 16, 32, 64]

        blocks = []
        prev = in_channels
        for out_ch in features:
            blocks.append(_conv_block(prev, out_ch))
            prev = out_ch
        self.backbone = nn.Sequential(*blocks)
        self.global_pool = nn.AdaptiveAvgPool3d(1)

        feat_dim = features[-1]

        def _head():
            return nn.Sequential(
                nn.Linear(feat_dim, fc_hidden),
                nn.ReLU(inplace=True),
                nn.Dropout(p=dropout),
                nn.Linear(fc_hidden, self.NUM_CLASSES),
            )

        self.head_sag = _head()
        self.head_fr  = _head()

    def forward(self, x: torch.Tensor):
        feat = self.backbone(x)
        feat = self.global_pool(feat).view(feat.size(0), -1)
        return self.head_sag(feat), self.head_fr(feat)

In [ ]:
model = TMJCondyleClassifier().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,} ({n_params / 1e6:.2f}M)")

with torch.no_grad():
    dummy = torch.randn(1, 1, TARGET_SIZE, TARGET_SIZE, TARGET_SIZE, device=device)
    out_sag, out_fr = model(dummy)
    print(f"Output shapes: sag={out_sag.shape}, fr={out_fr.shape}")

## 6. Training

In [ ]:
import datetime
import torch.optim as optim
from tqdm.notebook import tqdm

EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-5
LR_PATIENCE = 10
EARLY_STOPPING = 30
LABEL_SMOOTHING = 0.1


def compute_accuracy(out_sag, out_fr, labels: torch.Tensor):
    acc_sag = (out_sag.argmax(1) == labels[:, 0]).float().mean().item()
    acc_fr  = (out_fr.argmax(1)  == labels[:, 1]).float().mean().item()
    return {
        "acc_sagittal": acc_sag,
        "acc_frontal": acc_fr,
        "mean_accuracy": (acc_sag + acc_fr) / 2,
    }


def train_epoch(model, loader, criterions, optimizer, epoch):
    model.train()
    criterion_sag, criterion_fr = criterions
    running_loss = 0.0
    all_m = []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]")
    for volumes, labels in pbar:
        volumes, labels = volumes.to(device), labels.to(device)
        optimizer.zero_grad()
        out_sag, out_fr = model(volumes)
        loss = criterion_sag(out_sag, labels[:, 0]) + criterion_fr(out_fr, labels[:, 1])
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            m = compute_accuracy(out_sag, out_fr, labels)
        m["loss"] = loss.item()
        all_m.append(m)
        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{m['mean_accuracy']:.3f}")

    avg = {k: float(np.mean([m[k] for m in all_m])) for k in all_m[0]}
    avg["loss"] = running_loss / len(loader)
    return avg


def validate_epoch(model, loader, criterions, epoch):
    model.eval()
    criterion_sag, criterion_fr = criterions
    running_loss = 0.0
    all_m = []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Val]  ")
    with torch.no_grad():
        for volumes, labels in pbar:
            volumes, labels = volumes.to(device), labels.to(device)
            out_sag, out_fr = model(volumes)
            loss = criterion_sag(out_sag, labels[:, 0]) + criterion_fr(out_fr, labels[:, 1])

            m = compute_accuracy(out_sag, out_fr, labels)
            m["loss"] = loss.item()
            all_m.append(m)
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{m['mean_accuracy']:.3f}")

    avg = {k: float(np.mean([m[k] for m in all_m])) for k in all_m[0]}
    avg["loss"] = running_loss / len(loader)
    return avg

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
exp_dir = Path(OUTPUT_DIR) / f"condyle_v3_{timestamp}"
exp_dir.mkdir(parents=True, exist_ok=True)

config = {
    "epochs": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE, "crop_size": CROP_SIZE, "target_size": TARGET_SIZE,
    "lr_patience": LR_PATIENCE, "early_stopping": EARLY_STOPPING,
    "label_smoothing": LABEL_SMOOTHING,
    "train_crops": len(train_ds), "val_crops": len(val_ds),
    "train_studies": len(train_records), "val_studies": len(val_records),
    "device": str(device),
    "augmentations": ["shift", "flip_3d", "rot90", "gaussian_noise", "intensity_jitter"],
    "model_features": [8, 16, 32, 64],
    "model_dropout": 0.5,
}
with open(exp_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Experiment dir: {exp_dir}")
print(f"Config: {json.dumps(config, indent=2)}")

In [ ]:
from collections import Counter

sag_counts = Counter(r["sag"] for r in train_crop_records)
fr_counts = Counter(r["fr"] for r in train_crop_records)

def _class_weights(counts, num_classes=3):
    total = sum(counts.values())
    return torch.tensor(
        [total / (num_classes * max(counts.get(c, 0), 1)) for c in range(num_classes)],
        dtype=torch.float32,
    )

sag_w = _class_weights(sag_counts)
fr_w = _class_weights(fr_counts)
print(f"Sagittal counts: {dict(sorted(sag_counts.items()))} → weights {[round(x, 2) for x in sag_w.tolist()]}")
print(f"Frontal  counts: {dict(sorted(fr_counts.items()))} → weights {[round(x, 2) for x in fr_w.tolist()]}")

criterion_sag = nn.CrossEntropyLoss(weight=sag_w.to(device), label_smoothing=LABEL_SMOOTHING)
criterion_fr  = nn.CrossEntropyLoss(weight=fr_w.to(device), label_smoothing=LABEL_SMOOTHING)
criterions = (criterion_sag, criterion_fr)

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=LR_PATIENCE
)

best_val_acc = -1.0
best_model_path = exp_dir / "best_model.pth"
epochs_no_improve = 0
history = []

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    train_m = train_epoch(model, train_loader, criterions, optimizer, epoch)
    val_m = validate_epoch(model, val_loader, criterions, epoch)

    scheduler.step(val_m["mean_accuracy"])
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch}/{EPOCHS}  "
        f"Train loss={train_m['loss']:.4f} acc={train_m['mean_accuracy']:.3f}  "
        f"Val loss={val_m['loss']:.4f} acc={val_m['mean_accuracy']:.3f}  "
        f"LR={current_lr:.6f}"
    )

    row = {"epoch": epoch, "lr": current_lr}
    row.update({f"train_{k}": v for k, v in train_m.items()})
    row.update({f"val_{k}": v for k, v in val_m.items()})
    history.append(row)

    with open(exp_dir / "metrics.jsonl", "a") as f:
        f.write(json.dumps(row) + "\n")

    if val_m["mean_accuracy"] > best_val_acc:
        best_val_acc = val_m["mean_accuracy"]
        epochs_no_improve = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_accuracy": best_val_acc,
                "val_metrics": val_m,
            },
            best_model_path,
        )
        print(f"  >>> Saved best model (acc={best_val_acc:.3f})")
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{EARLY_STOPPING})")

    if EARLY_STOPPING > 0 and epochs_no_improve >= EARLY_STOPPING:
        print(f"\nEarly stopping at epoch {epoch}")
        break

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 60)
print(f"TRAINING COMPLETE — Best val accuracy: {best_val_acc:.3f}")
print(f"Artefacts: {exp_dir}")
print("=" * 60)

## 7. Visualization

In [ ]:
epochs_list = [h["epoch"] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_list, [h["train_loss"] for h in history], label="Train", linewidth=2)
axes[0].plot(epochs_list, [h["val_loss"] for h in history], label="Val", linewidth=2)
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_list, [h["train_mean_accuracy"] for h in history], label="Train", linewidth=2)
axes[1].plot(epochs_list, [h["val_mean_accuracy"] for h in history], label="Val", linewidth=2)
axes[1].axhline(y=best_val_acc, color="r", linestyle="--", alpha=0.5, label=f"Best={best_val_acc:.3f}")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Mean Accuracy")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

for name in HEAD_NAMES:
    axes[2].plot(epochs_list, [h[f"val_acc_{name}"] for h in history], label=name, linewidth=1.5)
axes[2].set(xlabel="Epoch", ylabel="Accuracy", title="Per-Head Val Accuracy")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(exp_dir / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {exp_dir / 'training_curves.png'}")

In [ ]:
# Learning rate schedule
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_list, [h["lr"] for h in history], linewidth=2, color="tab:orange")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("Learning Rate Schedule")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Export

Скопировать лучшую модель на Google Drive и/или скачать локально.

In [ ]:
import shutil

drive_dest = DRIVE_ROOT / "trained_models"
drive_dest.mkdir(parents=True, exist_ok=True)

dest_path = drive_dest / f"position_classifier_{timestamp}.pth"
shutil.copy2(best_model_path, dest_path)
print(f"Model saved to Google Drive: {dest_path}")

curves_src = exp_dir / "training_curves.png"
if curves_src.exists():
    shutil.copy2(curves_src, drive_dest / f"training_curves_{timestamp}.png")
    print(f"Curves saved to Google Drive")

metrics_src = exp_dir / "metrics.jsonl"
if metrics_src.exists():
    shutil.copy2(metrics_src, drive_dest / f"metrics_{timestamp}.jsonl")
    print(f"Metrics saved to Google Drive")

In [ ]:
from google.colab import files

files.download(str(best_model_path))

In [ ]:
ckpt = torch.load(best_model_path, map_location="cpu", weights_only=False)
print(f"Best epoch:        {ckpt['epoch']}")
print(f"Best val accuracy: {ckpt['best_val_accuracy']:.3f}")
print(f"Val metrics:")
for k, v in ckpt["val_metrics"].items():
    print(f"  {k}: {v:.4f}")
print(f"\nModel class: TMJCondyleClassifier (2 heads: sagittal + frontal)")
print(f"Input: (1, {TARGET_SIZE}, {TARGET_SIZE}, {TARGET_SIZE})")

## 9. Training Analysis

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# ---- 1. History → DataFrame ----
df = pd.DataFrame(history)
display(df.style.format(precision=4).background_gradient(
    subset=["val_mean_accuracy"], cmap="Greens",
).background_gradient(
    subset=["val_loss"], cmap="Reds_r",
))

# ---- 2. Best / worst epochs ----
best_row = df.loc[df["val_mean_accuracy"].idxmax()]
worst_row = df.loc[df["val_mean_accuracy"].idxmin()]
print(f"Best  epoch {int(best_row['epoch'])}: val_acc={best_row['val_mean_accuracy']:.4f}  val_loss={best_row['val_loss']:.4f}")
print(f"Worst epoch {int(worst_row['epoch'])}: val_acc={worst_row['val_mean_accuracy']:.4f}  val_loss={worst_row['val_loss']:.4f}")

# ---- 3. Overfit gap ----
df["overfit_gap"] = df["train_mean_accuracy"] - df["val_mean_accuracy"]
print(f"\nOverfit gap (last 5 epochs): {df['overfit_gap'].tail(5).mean():.4f}")
print(f"Max overfit gap:             {df['overfit_gap'].max():.4f} (epoch {int(df.loc[df['overfit_gap'].idxmax(), 'epoch'])})")

# ---- 4. Detailed plots ----
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# Loss
axes[0, 0].plot(df["epoch"], df["train_loss"], label="Train", linewidth=2)
axes[0, 0].plot(df["epoch"], df["val_loss"], label="Val", linewidth=2)
axes[0, 0].set(title="Loss", xlabel="Epoch", ylabel="Loss")
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# Mean accuracy
axes[0, 1].plot(df["epoch"], df["train_mean_accuracy"], label="Train", linewidth=2)
axes[0, 1].plot(df["epoch"], df["val_mean_accuracy"], label="Val", linewidth=2)
axes[0, 1].axhline(best_row["val_mean_accuracy"], color="r", ls="--", alpha=0.5,
                    label=f"Best={best_row['val_mean_accuracy']:.3f}")
axes[0, 1].set(title="Mean Accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

# Per-head val accuracy
for name in HEAD_NAMES:
    axes[0, 2].plot(df["epoch"], df[f"val_acc_{name}"], label=name, linewidth=1.5)
axes[0, 2].set(title="Val Accuracy per Head", xlabel="Epoch", ylabel="Accuracy")
axes[0, 2].legend(); axes[0, 2].grid(alpha=0.3)

# Overfit gap
axes[1, 0].plot(df["epoch"], df["overfit_gap"], linewidth=2, color="tab:red")
axes[1, 0].axhline(0, color="gray", ls="--", alpha=0.5)
axes[1, 0].set(title="Overfit Gap (train_acc - val_acc)", xlabel="Epoch", ylabel="Gap")
axes[1, 0].grid(alpha=0.3)

# LR schedule
axes[1, 1].plot(df["epoch"], df["lr"], linewidth=2, color="tab:orange")
axes[1, 1].set(title="Learning Rate", xlabel="Epoch", ylabel="LR")
axes[1, 1].set_yscale("log"); axes[1, 1].grid(alpha=0.3)

# Loss ratio
df["loss_ratio"] = df["val_loss"] / df["train_loss"].clip(lower=1e-6)
axes[1, 2].plot(df["epoch"], df["loss_ratio"], linewidth=2, color="tab:purple")
axes[1, 2].axhline(1.0, color="gray", ls="--", alpha=0.5)
axes[1, 2].set(title="Val/Train Loss Ratio", xlabel="Epoch", ylabel="Ratio")
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(exp_dir / "analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- 5. Confusion matrices on val set (best model) ----
ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best model from epoch {ckpt['epoch']} (val_acc={ckpt['best_val_accuracy']:.3f})")
del ckpt
model.eval()
all_preds_sag, all_preds_fr = [], []
all_true_sag, all_true_fr = [], []

with torch.no_grad():
    for volumes, labels in val_loader:
        volumes = volumes.to(device)
        out_sag, out_fr = model(volumes)
        all_preds_sag.extend(out_sag.argmax(1).cpu().tolist())
        all_preds_fr.extend(out_fr.argmax(1).cpu().tolist())
        all_true_sag.extend(labels[:, 0].tolist())
        all_true_fr.extend(labels[:, 1].tolist())

CLASS_LABELS = [0, 1, 2]
CLASS_NAMES = ["Anterior/Medial", "Normal", "Posterior/Lateral"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, true, pred, title in [
    (axes[0], all_true_sag, all_preds_sag, "Sagittal"),
    (axes[1], all_true_fr,  all_preds_fr,  "Frontal"),
]:
    cm = confusion_matrix(true, pred, labels=CLASS_LABELS)
    present = sorted(set(true) | set(pred))
    names = [CLASS_NAMES[i] for i in present]
    cm_present = confusion_matrix(true, pred, labels=present)
    sns.heatmap(cm_present, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=names, yticklabels=names)
    ax.set(title=f"{title} — Confusion Matrix", xlabel="Predicted", ylabel="True")

plt.tight_layout()
plt.savefig(exp_dir / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- 6. Classification reports ----
print("=" * 60)
print("SAGITTAL classification report")
print("=" * 60)
print(classification_report(all_true_sag, all_preds_sag, labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0))

print("=" * 60)
print("FRONTAL classification report")
print("=" * 60)
print(classification_report(all_true_fr, all_preds_fr, labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0))

# ---- 7. Summary ----
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Total epochs:       {len(history)}")
print(f"Best val accuracy:  {best_row['val_mean_accuracy']:.4f} (epoch {int(best_row['epoch'])})")
print(f"  - Sagittal:       {best_row['val_acc_sagittal']:.4f}")
print(f"  - Frontal:        {best_row['val_acc_frontal']:.4f}")
print(f"Final LR:           {df['lr'].iloc[-1]:.2e}")
print(f"Overfit gap (last): {df['overfit_gap'].tail(5).mean():.4f}")
print(f"Train samples:      {len(train_ds)} crops")
print(f"Val samples:        {len(val_ds)} crops")
print(f"Model:              TMJCondyleClassifier (2 heads)")
print(f"Input:              (1, {TARGET_SIZE}, {TARGET_SIZE}, {TARGET_SIZE})")
print(f"Artefacts:          {exp_dir}")

In [ ]:
# Сохранить все результаты в один JSON и скачать
analysis_report = {
    "config": config,
    "total_epochs": len(history),
    "best_epoch": int(best_row["epoch"]),
    "best_val_accuracy": round(float(best_row["val_mean_accuracy"]), 4),
    "best_val_acc_sagittal": round(float(best_row["val_acc_sagittal"]), 4),
    "best_val_acc_frontal": round(float(best_row["val_acc_frontal"]), 4),
    "best_val_loss": round(float(best_row["val_loss"]), 4),
    "final_lr": float(df["lr"].iloc[-1]),
    "overfit_gap_last5": round(float(df["overfit_gap"].tail(5).mean()), 4),
    "max_overfit_gap": round(float(df["overfit_gap"].max()), 4),
    "history": history,
    "classification_report": {
        "sagittal": classification_report(
            all_true_sag, all_preds_sag,
            labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0, output_dict=True,
        ),
        "frontal": classification_report(
            all_true_fr, all_preds_fr,
            labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0, output_dict=True,
        ),
    },
    "confusion_matrix": {
        "sagittal": confusion_matrix(all_true_sag, all_preds_sag, labels=[0,1,2]).tolist(),
        "frontal":  confusion_matrix(all_true_fr,  all_preds_fr,  labels=[0,1,2]).tolist(),
    },
    "val_predictions": {
        "sagittal": {"true": all_true_sag, "pred": all_preds_sag},
        "frontal":  {"true": all_true_fr,  "pred": all_preds_fr},
    },
}

report_path = exp_dir / "training_analysis.json"
with open(report_path, "w") as f:
    json.dump(analysis_report, f, indent=2, ensure_ascii=False)

from google.colab import files
files.download(str(report_path))
print(f"Saved & downloading: {report_path}")